# Stuff+ / Location+ / Pitching+ — Diablos Rojos (ISAC 2026)

**Entregable 1:** Stuff+ por pitcheo, pitcher y arsenal, y comparación CDMX (altitud extrema) vs. otros estadios.

| Paso | Qué hace |
|---|---|
| 0 | Configuración y carga de datos |
| 1 | Valor en carreras de cada pitcheo (según el conteo) |
| 2 | Variables: físicas, de arsenal y de contexto |
| 3 | Variable objetivo descompuesta: 5 resultados posibles de un pitcheo + calidad de la bola en juego |
| 4 | Entrenamiento XGBoost con validación *pitcher holdout* (predicciones fuera de muestra para cada pitcheo) |
| 5 | Carreras esperadas (xRV) → escala 100 + 10·z |
| 6 | Validación: calidad, estabilidad, capacidad predictiva, por temporada y por altitud |
| 7 | Efecto altitud: el mismo pitcheo calificado en cada nivel de altitud |
| 8 | Agregación por pitcher y por arsenal, y exportación |
| 9 | Subir los resultados al API 2 para que la web app los use |

**Kernel:** Python 3.12 (`/usr/local/bin/python3`), no anaconda3.

**Tiempo:** la primera vez el Paso 4 entrena 30+ modelos (~30–40 min). Los resultados se guardan en la carpeta `modelos/`; las siguientes veces se cargan en segundos. Para volver a entrenar, pon `REENTRENAR = True`.

## Paso 0: Configuración

In [ ]:
import os                                    # para crear carpetas y revisar si existen archivos
import time                                  # para medir cuánto tarda cada entrenamiento
import numpy as np                           # cálculo numérico (arreglos, raíces, senos...)
import pandas as pd                          # tablas de datos (DataFrames)
import xgboost as xgb                        # el modelo: gradient boosting de árboles
import matplotlib.pyplot as plt              # gráficas
from sklearn.model_selection import GroupKFold    # divide los datos en bloques sin repetir pitchers
from sklearn.metrics import log_loss, mean_squared_error   # métricas de error

pd.set_option('display.max_columns', 60)     # mostrar hasta 60 columnas al imprimir tablas
pd.set_option('display.width', 250)          # tablas anchas sin cortar renglones

CARPETA = os.path.expanduser("~/Desktop/hackathon")   # carpeta de DATOS, fuera del repo (es público); ~ = tu usuario
CARPETA_MODELOS = os.path.join(CARPETA, "modelos")         # aquí se guardan modelos y predicciones
CARPETA_RESULTADOS = os.path.join(CARPETA, "resultados")   # aquí se exportan las tablas finales
os.makedirs(CARPETA_MODELOS, exist_ok=True)                # crea la carpeta si no existe (y no falla si ya existe)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)

REENTRENAR = False    # False: carga los modelos guardados si existen. True: entrena todo de nuevo.
SEMILLA = 42          # fija el azar para que los resultados sean reproducibles

df = pd.read_pickle(os.path.join(CARPETA, "stuff_model_df.pkl"))   # carga los 635 mil pitcheos
print(df.shape, "| xgboost", xgb.__version__)                       # (renglones, columnas) y versión

## Paso 1: Valor en carreras de cada pitcheo

Cada conteo (bolas-strikes) tiene un **valor esperado en carreras** para el bateador: en 3-0 va ganando, en 0-2 va perdiendo. El valor de un pitcheo es **cuánto movió ese valor**:

- Si **no termina** la aparición al bat: `valor(conteo siguiente) − valor(conteo actual)`
- Si **la termina**: `valor(evento) − valor(conteo actual)`

Signo: **positivo = bueno para el bateador**. En el Paso 5 lo invertimos para que en Stuff+ "más alto = mejor para el pitcher".

**Por qué así:** es el mismo principio del *delta run expectancy* de Statcast. No tenemos corredores en base (no vienen en los datos), así que usamos solo el conteo. Es una limitación que reportamos.

### 1a. Identificar cada aparición al bat (`pa_id`)
Los datos vienen en orden cronológico. Un pitcheo **termina** la aparición si fue ponche, base por bolas, bola en juego o golpe. Una aparición **nueva empieza** si cambió el juego, la entrada, la mitad de entrada o el bateador respecto al pitcheo anterior, o si el pitcheo anterior terminó una aparición.

In [ ]:
# ¿Este pitcheo TERMINA la aparición al bat?
# True si fue ponche o base por bolas (columna KorBB), o si la bola quedó en juego o golpeó al bateador (PitchCall).
termina_pa = (
    df['KorBB'].isin(['Strikeout', 'Walk'])
    | df['PitchCall'].isin(['InPlay', 'HitByPitch'])
)

# ¿Este pitcheo EMPIEZA una aparición nueva?
# .shift() baja la columna un renglón: así cada pitcheo "ve" el valor del pitcheo anterior y los comparamos.
empieza_pa = (
    (df['game_anon_id'] != df['game_anon_id'].shift())          # cambió el juego
    | (df['Inning'] != df['Inning'].shift())                    # cambió la entrada
    | (df['Top/Bottom'] != df['Top/Bottom'].shift())            # cambió la mitad (alta/baja)
    | (df['batter_anon_id'] != df['batter_anon_id'].shift())    # cambió el bateador
    | termina_pa.shift(fill_value=False)                        # el pitcheo anterior cerró la aparición
)

# cumsum (suma acumulada) sobre True/False: cada True suma 1, así todos los pitcheos
# de la misma aparición comparten el mismo número.
df['pa_id'] = empieza_pa.cumsum()
print(f"Apariciones al bat: {df['pa_id'].nunique():,}")   # esperado: 168,332

### 1b. Evento final y su valor en carreras
Usamos *linear weights* estándar: cuántas carreras vale, en promedio, cada tipo de evento respecto a un out. Son valores de MLB. La LMB es más ofensiva, así que es otra limitación a reportar; lo importante para Stuff+ es el **orden relativo** entre eventos, no el valor exacto.

In [ ]:
# Valor en carreras de cada evento que termina una aparición al bat
VALOR_EVENTO = {'Walk': 0.31, 'HitByPitch': 0.33, 'Single': 0.46, 'Error': 0.46,
                'Double': 0.77, 'Triple': 1.05, 'HomeRun': 1.40,
                'Out': -0.27, 'Strikeout': -0.27, 'FieldersChoice': -0.27, 'Sacrifice': -0.27}

# np.select revisa las condiciones EN ORDEN y asigna la opción de la primera que se cumple
df['evento'] = np.select(
    [df['KorBB'] == 'Strikeout',          # condición 1 → 'Strikeout'
     df['KorBB'] == 'Walk',               # condición 2 → 'Walk'
     df['PitchCall'] == 'HitByPitch',     # condición 3 → 'HitByPitch'
     df['PitchCall'] == 'InPlay'],        # condición 4 → lo que diga play_result (Single, Out, HomeRun...)
    ['Strikeout', 'Walk', 'HitByPitch', df['play_result']],
    default=None,                         # ninguna condición: no es un evento final
)
df['evento'] = df['evento'].where(termina_pa)      # solo cuenta en el pitcheo que termina la aparición
df['rv_evento'] = df['evento'].map(VALOR_EVENTO)   # traduce el nombre del evento a carreras

df['evento'].value_counts()                        # esperado: Out 66,920 · Strikeout 30,648 · Single 28,579

### 1c. Valor de cada conteo
El valor de un conteo, por ejemplo 1-2, es el **promedio del valor final de todas las apariciones que pasaron por 1-2**. Cada aparición cuenta **una sola vez** por conteo, aunque haya tenido varios fouls en 1-2.

In [ ]:
# 1) Valor final de cada aparición = el rv_evento de su último pitcheo.
#    dropna quita los pitcheos sin evento; groupby('pa_id').last() toma el último valor de cada aparición.
valor_final_pa = df.dropna(subset=['rv_evento']).groupby('pa_id')['rv_evento'].last()

# 2) A cada pitcheo le pegamos el valor final de SU aparición (map busca el pa_id en valor_final_pa).
df['rv_pa'] = df['pa_id'].map(valor_final_pa)

# 3) Solo apariciones con evento final (sin juegos cortados) y conteos posibles (máx. 3 bolas, 2 strikes).
conteo_valido = df['rv_pa'].notna() & (df['Balls'] <= 3) & (df['Strikes'] <= 2)

# 4) Promedio por conteo: drop_duplicates deja una sola fila por (aparición, bolas, strikes).
valor_conteo = (
    df[conteo_valido]
    .drop_duplicates(['pa_id', 'Balls', 'Strikes'])
    .groupby(['Balls', 'Strikes'])['rv_pa']
    .mean()
)
valor_conteo.unstack().round(3)    # unstack pone los strikes como columnas: tabla de 4 × 3

### 1d. Valor de cada pitcheo (`rv_pitcheo`)
Revisión de sentido común: el promedio de toda la liga debe dar **≈ 0** (lo que gana el pitcher lo pierde el bateador), el *swinging strike* debe ser el pitcheo más negativo, y un foul con 2 strikes debe valer 0 porque el conteo no cambia.

In [ ]:
# 1) Valor del conteo ACTUAL de cada pitcheo.
#    valor_conteo tiene índice doble (Balls, Strikes); convertimos cada pitcheo en su pareja (Balls, Strikes)
#    con MultiIndex.from_frame y la buscamos con map.
df['v_conteo'] = pd.MultiIndex.from_frame(df[['Balls', 'Strikes']]).map(valor_conteo)

# 2) Valor del conteo SIGUIENTE: el v_conteo del próximo pitcheo DENTRO de la misma aparición.
#    shift(-1) sube la columna un renglón; el groupby evita "brincar" a la siguiente aparición.
df['v_conteo_sig'] = df.groupby('pa_id')['v_conteo'].shift(-1)

# 3) np.where(condición, si_verdadero, si_falso), renglón por renglón
df['rv_pitcheo'] = np.where(
    termina_pa,
    df['rv_evento'] - df['v_conteo'],       # termina: evento − conteo actual
    df['v_conteo_sig'] - df['v_conteo'],    # no termina: conteo siguiente − conteo actual
)

print(f"Promedio general: {df['rv_pitcheo'].mean():.3f}")     # esperado ≈ 0.000
df.groupby('PitchCall')['rv_pitcheo'].agg(['mean', 'size']).round(3)

## Paso 2: Variables

### 2a. Qué pitcheos entran al modelo
- **Tipos:** los 7 con muestra suficiente. El `Sweeper` (588 pitcheos, solo en un estadio de altitud extrema) se reetiqueta como `Slider`: es un slider de barrido, y así no queda un tipo que solo existe en un nivel de altitud. `Other`, `OneSeamFastBall`, `TwoSeamFastBall` y `Knuckleball` (64 pitcheos en total) se excluyen.
- **Mano:** se quitan pitchers y bateadores `Undefined` o `Switch` (~550 pitcheos).
- **Resultado:** se quitan los `PitchCall == 'Undefined'`, los pitcheos sin valor en carreras (último pitcheo de juegos cortados) y los que no tienen ID de pitcher.

El pitcheo anterior (`pitcheo_anterior`) se calcula **antes** de filtrar, para que un pitcheo excluido no rompa la secuencia.

In [ ]:
TIPOS_MODELO = ['Four-Seam', 'Sinker', 'Cutter', 'Slider', 'Curveball', 'Changeup', 'Splitter']

# AutoPitchType viene como categoría; astype(str) lo vuelve texto para poder reetiquetar libremente
df['tipo'] = df['AutoPitchType'].astype(str).replace({'Sweeper': 'Slider'})     # sweeper → slider

# Pitcheo anterior dentro de la misma aparición. El primero de cada aparición no tiene anterior → 'Primero'.
df['pitcheo_anterior'] = df.groupby('pa_id')['tipo'].shift(1).fillna('Primero')

valido = (
    df['tipo'].isin(TIPOS_MODELO)
    & df['PitcherThrows'].isin(['Left', 'Right'])
    & df['BatterSide'].isin(['Left', 'Right'])
    & (df['PitchCall'] != 'Undefined')
    & df['rv_pitcheo'].notna()
    & df['pitcher_anon_id'].notna()          # ~3 mil pitcheos sin ID de pitcher
)
d = df[valido].copy().reset_index(drop=True)    # d = datos del modelo; reset_index numera 0..n-1
print(f"Pitcheos en el modelo: {len(d):,} de {len(df):,} ({len(d) / len(df):.1%})")

### 2b. Variables físicas (reflejadas para zurdos)
Trackman mide el quiebre horizontal, el punto de salida y el eje de giro **desde la perspectiva del cátcher**, así que un slider zurdo y uno derecho con la misma forma tienen signos opuestos. Los reflejamos para que **positivo = hacia el lado del brazo** en ambos casos. Así el modelo aprende cada forma de pitcheo una vez y no dos.

El **eje de giro** es un ángulo (0–360°): 359° está junto a 1°, pero numéricamente parecen lejanos. Por eso lo convertimos en **seno y coseno**, que tratan el círculo correctamente.

In [ ]:
zurdo = d['PitcherThrows'] == 'Left'                                   # True para pitchers zurdos

d['hb'] = np.where(zurdo, -d['HorzBreak'], d['HorzBreak'])             # quiebre horizontal hacia el brazo
d['rel_side'] = np.where(zurdo, -d['RelSide'], d['RelSide'])           # punto de salida lateral
eje = np.where(zurdo, 360 - d['SpinAxis'], d['SpinAxis'])              # eje de giro reflejado (espejo)
d['eje_sin'] = np.sin(np.radians(eje))                                 # componente seno del ángulo
d['eje_cos'] = np.cos(np.radians(eje))                                 # componente coseno del ángulo

d[['hb', 'rel_side', 'eje_sin', 'eje_cos']].describe().round(2)

### 2c. Variables de arsenal (relativas a la recta primaria)
FanGraphs subraya que los pitcheos secundarios se juzgan **contra la recta**: un changeup de 84 mph es muy distinto si la recta va a 96 o a 88.

1. **Recta primaria** de cada pitcher-temporada: la más usada entre Four-Seam y Sinker. Si no tiene ninguna, su Cutter.
2. **Promedio de la recta del mismo juego** (mismo estadio, mismo aire, mismo día). Si ese día no tiró recta, se usa su promedio de la temporada. Así la diferencia "slider − recta" en CDMX compara dos pitcheos en el mismo aire.
3. **Diferencias:** velocidad, IVB y HB contra la recta. Distancia del punto de salida contra el de la recta (**túnel**: menor distancia = los pitcheos se ven iguales al salir de la mano). Y el % de uso del pitcheo.

In [ ]:
# --- 1) Recta primaria por pitcher-temporada
conteo_tipos = d.groupby(['pitcher_anon_id', 'year'])['tipo'].value_counts().rename('n').reset_index()

def elegir_recta(g):
    """g = los tipos de un pitcher-temporada con su conteo. Devuelve su recta primaria."""
    for candidatos in (['Four-Seam', 'Sinker'], ['Cutter']):          # primero rectas, luego cutter
        x = g[g['tipo'].isin(candidatos)]
        if len(x):
            return x.sort_values('n')['tipo'].iloc[-1]                 # la más usada (última al ordenar)
    return None                                                        # pitcher sin recta ni cutter

recta_primaria = (conteo_tipos.groupby(['pitcher_anon_id', 'year'])
                              .apply(elegir_recta, include_groups=False)
                              .rename('fb_tipo'))
d = d.join(recta_primaria, on=['pitcher_anon_id', 'year'])            # pega fb_tipo a cada pitcheo
print("Recta primaria por pitcher-temporada:", recta_primaria.value_counts(dropna=False).to_dict())

# --- 2) Promedios de la recta: por juego y, como respaldo, por temporada
COLS_RECTA = {'RelSpeed': 'fb_velo', 'InducedVertBreak': 'fb_ivb', 'hb': 'fb_hb',
              'RelHeight': 'fb_rel_h', 'rel_side': 'fb_rel_s'}
es_recta = d['tipo'] == d['fb_tipo']                                   # pitcheos que SON su recta primaria

fb_juego = d[es_recta].groupby(['pitcher_anon_id', 'game_anon_id'])[list(COLS_RECTA)].mean().rename(columns=COLS_RECTA)
fb_temp = d[es_recta].groupby(['pitcher_anon_id', 'year'])[list(COLS_RECTA)].mean().rename(columns=COLS_RECTA)

d = d.join(fb_juego, on=['pitcher_anon_id', 'game_anon_id'])          # promedio del juego
respaldo = d[['pitcher_anon_id', 'year']].join(fb_temp, on=['pitcher_anon_id', 'year'])
for c in COLS_RECTA.values():
    d[c] = d[c].fillna(respaldo[c])                                    # si falta el del juego, el de la temporada
print(f"Pitcheos sin referencia de recta: {d['fb_velo'].isna().mean():.2%} (XGBoost maneja los NaN)")

# --- 3) Diferencias contra la recta, túnel y uso
d['velo_diff_vs_fb'] = d['RelSpeed'] - d['fb_velo']
d['ivb_diff_vs_fb'] = d['InducedVertBreak'] - d['fb_ivb']
d['hb_diff_vs_fb'] = d['hb'] - d['fb_hb']
d['dist_salida_fb'] = np.hypot(d['RelHeight'] - d['fb_rel_h'], d['rel_side'] - d['fb_rel_s'])   # pies
d['uso_pct'] = (d.groupby(['pitcher_anon_id', 'year', 'tipo'])['PitchUID'].transform('size')
                / d.groupby(['pitcher_anon_id', 'year'])['PitchUID'].transform('size'))

(d.groupby('tipo')[['velo_diff_vs_fb', 'ivb_diff_vs_fb', 'hb_diff_vs_fb', 'dist_salida_fb', 'uso_pct']]
  .median().round(2).loc[TIPOS_MODELO])

### 2d. Variables de contexto
- **Altitud ordinal:** No = 0, Medium = 1, Extreme = 2. El orden le dice al modelo que "Medium" está entre las otras dos.
- **Ubicación relativa al bateador:** `PlateLocSide` también se mide desde el cátcher. La reflejamos según el lado del bateador para que **`loc_x` positivo = lejos del bateador (afuera)** y negativo = pegado (adentro). La comprobación es que los pelotazos deben tener `loc_x` negativo.
- **`mismo_lado`:** 1 si pitcher y bateador son de la misma mano (derecho vs. derecho), la ventaja clásica del pitcher (*platoon*).
- **Tipo de pitcheo y pitcheo anterior** como categorías: XGBoost las maneja directamente.

In [ ]:
d['altitud'] = d['altitude_category'].astype(str).map({'No Altitude': 0, 'Medium Altitude': 1, 'Extreme Altitude': 2})
d['mano_derecha'] = (d['PitcherThrows'] == 'Right').astype(int)
d['mismo_lado'] = (d['PitcherThrows'] == d['BatterSide']).astype(int)
# Para el bateador derecho se invierte el signo y para el zurdo no. Así loc_x positivo = lejos del bateador.
# Comprobación empírica abajo: los pelotazos deben quedar en negativo (pegados al bateador).
d['loc_x'] = np.where(d['BatterSide'] == 'Right', -d['PlateLocSide'], d['PlateLocSide'])
d['tipo'] = pd.Categorical(d['tipo'], categories=TIPOS_MODELO)
d['pitcheo_anterior'] = pd.Categorical(d['pitcheo_anterior'], categories=TIPOS_MODELO + ['Primero', 'Other', 'Sweeper',
                                                                                         'OneSeamFastBall', 'TwoSeamFastBall', 'Knuckleball'])

print("loc_x promedio de los pelotazos (debe ser negativo = adentro):",
      round(d.loc[d['PitchCall'] == 'HitByPitch', 'loc_x'].mean(), 2))
print(d['altitud'].value_counts().sort_index().to_dict())

### 2e. Qué variables usa cada modelo

| Capa | Pregunta | Variables |
|---|---|---|
| **Stuff+** | ¿Qué tan bueno es el pitcheo por **cómo sale de la mano**? | 9 físicas + 5 de arsenal + mano + altitud |
| **Location+** | ¿Qué tan bien **ubicado** está, dado el tipo y el conteo? | ubicación + conteo + tipo + mismo_lado + mano + altitud |
| **Pitching+** | Todo junto | Stuff + Location + pitcheo anterior + outs + entrada |

**Stuff+ no usa la etiqueta de tipo de pitcheo**, solo su forma física ("*shape over label*", hipótesis H4 del reto). Así un slider que se mueve como cutter se califica como lo que es.

In [ ]:
FISICAS = ['RelSpeed', 'SpinRate', 'eje_sin', 'eje_cos', 'InducedVertBreak', 'hb',
           'Extension', 'RelHeight', 'rel_side']
ARSENAL = ['velo_diff_vs_fb', 'ivb_diff_vs_fb', 'hb_diff_vs_fb', 'dist_salida_fb', 'uso_pct']
UBICACION = ['loc_x', 'PlateLocHeight']
CONTEO = ['Balls', 'Strikes']

VARIABLES = {
    'stuff':    FISICAS + ARSENAL + ['mano_derecha', 'altitud'],
    'location': UBICACION + CONTEO + ['tipo', 'mismo_lado', 'mano_derecha', 'altitud'],
    'pitching': FISICAS + ARSENAL + UBICACION + CONTEO + ['tipo', 'mismo_lado', 'mano_derecha', 'altitud',
                                                          'pitcheo_anterior', 'Outs', 'Inning'],
}
for capa, cols in VARIABLES.items():
    print(f"{capa:9s} {len(cols):2d} variables")

## Paso 3: Variable objetivo descompuesta

Predecir directamente las carreras de cada pitcheo es muy ruidoso: un mismo pitcheo excelente puede terminar en jonrón por suerte. Por eso lo **descomponemos**:

1. **Modelo de resultado** (clasificación, 5 clases): P(bola), P(strike cantado), P(whiff), P(foul), P(en juego). Los pelotazos cuentan como bola.
2. **Modelo de bola en juego** (regresión): si la ponen en juego, ¿cuántas carreras se esperan? Se entrena solo con bolas en juego.
3. **Carreras esperadas del pitcheo:**

$$xRV = \sum_{k} P(k) \cdot RV_k$$

donde $RV_k$ es el valor en carreras del resultado $k$. Para Location+ y Pitching+ ese valor depende del conteo (una bola en 3-2 es base por bolas; un foul en 0-2 vale 0). Para Stuff+, que no ve el conteo, usamos el promedio de la liga.

Esto da los P(whiff), P(strike cantado), etc. que pide el reto, y es mucho más estable que el resultado observado.

In [ ]:
CLASES = ['bola', 'strike_cantado', 'whiff', 'foul', 'en_juego']
MAPA_RESULTADO = {'BallCalled': 'bola', 'BallinDirt': 'bola', 'HitByPitch': 'bola',
                  'StrikeCalled': 'strike_cantado', 'StrikeSwinging': 'whiff',
                  'FoulBall': 'foul', 'FoulBallNotFieldable': 'foul', 'FoulBallFieldable': 'foul',
                  'InPlay': 'en_juego'}

d['resultado'] = d['PitchCall'].map(MAPA_RESULTADO)                      # nombre de la clase
d['y'] = d['resultado'].map({c: i for i, c in enumerate(CLASES)})        # número de clase 0..4 (lo que usa XGBoost)
en_juego = (d['resultado'] == 'en_juego') & d['rv_evento'].notna()       # bolas en juego con evento válido

# Valor en carreras de cada resultado, por conteo (para Location+ y Pitching+)
rv_clase_conteo = d.groupby(['resultado', 'Balls', 'Strikes'])['rv_pitcheo'].mean()
# Valor promedio de cada resultado en toda la liga (para Stuff+)
rv_clase_liga = d.groupby('resultado')['rv_pitcheo'].mean()
# Valor del conteo promedio en que la bola se pone en juego (para Stuff+)
v_conteo_bip_liga = d.loc[en_juego, 'v_conteo'].mean()

print(d['resultado'].value_counts(normalize=True).round(3).to_dict())
rv_clase_conteo.unstack(['Balls', 'Strikes']).round(3)

## Paso 4: Entrenamiento con *pitcher holdout*

**GroupKFold por pitcher (5 bloques):** dividimos a los **pitchers** en 5 grupos. Para cada grupo entrenamos con los otros 4 y predecimos sus pitcheos. Así **cada pitcheo recibe una predicción de un modelo que nunca vio a ese pitcher**, que es justo lo que pide el reto con *pitcher holdout* y evita que el modelo "memorice" pitchers.

Dentro de cada entrenamiento separamos otro 10% de pitchers para **early stopping**: el modelo deja de agregar árboles cuando ya no mejora en pitchers que no ve.

**Hiperparámetros:**
- `max_depth=6`: interacciones de hasta 6 variables (velocidad × quiebre × altitud...).
- `learning_rate=0.1`: cada árbol corrige 10% del error restante.
- `min_child_weight=50`: cada hoja necesita suficientes pitcheos. Es regularización, porque los resultados de un pitcheo son muy ruidosos.
- `subsample` / `colsample_bytree=0.8`: cada árbol ve 80% de pitcheos y variables, lo que reduce el sobreajuste.
- `max_bin=128`: agrupa valores continuos en 128 intervalos, que lo hace más rápido.

In [ ]:
PARAMS_BASE = dict(
    n_estimators=1000,            # máximo de árboles (early stopping corta antes)
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=50,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',           # algoritmo rápido por histogramas
    max_bin=128,
    enable_categorical=True,      # permite columnas categóricas (tipo, pitcheo_anterior)
    early_stopping_rounds=30,     # para si 30 árboles seguidos no mejoran
    n_jobs=4,                     # usa los 4 núcleos de tu Mac
    random_state=SEMILLA,
)

grupos = d['pitcher_anon_id'].to_numpy()                        # a qué pitcher pertenece cada pitcheo
bloques = list(GroupKFold(n_splits=5).split(d, groups=grupos))  # 5 × (índices_entrenamiento, índices_prueba)


def separar_early_stopping(idx_entrena, fraccion=0.1, semilla=SEMILLA):
    """Aparta ~10% de los PITCHERS del entrenamiento para decidir cuándo parar de agregar árboles."""
    rng = np.random.default_rng(semilla)
    pitchers = np.unique(grupos[idx_entrena])
    apartados = rng.choice(pitchers, size=int(len(pitchers) * fraccion), replace=False)
    es_val = np.isin(grupos[idx_entrena], apartados)
    return idx_entrena[~es_val], idx_entrena[es_val]


def entrenar_modelo(X, y, idx_entrena, tipo):
    """Entrena un XGBoost (clasificador de 5 clases o regresor) con early stopping por pitchers."""
    idx_fit, idx_val = separar_early_stopping(idx_entrena)
    if tipo == 'resultado':
        modelo = xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', **PARAMS_BASE)
    else:
        modelo = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', **PARAMS_BASE)
    modelo.fit(X.iloc[idx_fit], y[idx_fit], eval_set=[(X.iloc[idx_val], y[idx_val])], verbose=False)
    return modelo


def entrenar_capa(capa):
    """Para una capa (stuff/location/pitching) entrena 5 modelos de resultado y 5 de bola en juego,
    uno por bloque, y guarda las predicciones fuera de muestra (OOF) de cada pitcheo."""
    X = d[VARIABLES[capa]]
    y_res = d['y'].to_numpy()
    y_bip = d['rv_evento'].to_numpy()
    idx_bip = np.flatnonzero(en_juego.to_numpy())                # posiciones de las bolas en juego
    prob_oof = np.zeros((len(d), len(CLASES)), dtype=np.float32)
    bip_oof = np.zeros(len(d), dtype=np.float32)
    modelos = []
    for i, (entrena, prueba) in enumerate(bloques):
        t0 = time.time()
        m_res = entrenar_modelo(X, y_res, entrena, 'resultado')
        m_bip = entrenar_modelo(X, y_bip, np.intersect1d(entrena, idx_bip), 'bip')   # solo bolas en juego
        prob_oof[prueba] = m_res.predict_proba(X.iloc[prueba])   # P de cada clase para pitchers no vistos
        bip_oof[prueba] = m_bip.predict(X.iloc[prueba])          # carreras esperadas si la ponen en juego
        m_res.save_model(os.path.join(CARPETA_MODELOS, f"{capa}_resultado_{i}.json"))
        m_bip.save_model(os.path.join(CARPETA_MODELOS, f"{capa}_bip_{i}.json"))
        modelos.append((m_res, m_bip))
        print(f"  {capa} bloque {i + 1}/5: {m_res.best_iteration + 1} árboles resultado, "
              f"{m_bip.best_iteration + 1} árboles bip, {time.time() - t0:.0f}s")
    np.savez(os.path.join(CARPETA_MODELOS, f"{capa}_oof.npz"), prob=prob_oof, bip=bip_oof)
    return prob_oof, bip_oof, modelos


def cargar_capa(capa):
    """Carga predicciones OOF y los 10 modelos guardados de una capa."""
    arch = np.load(os.path.join(CARPETA_MODELOS, f"{capa}_oof.npz"))
    modelos = []
    for i in range(len(bloques)):
        m_res, m_bip = xgb.XGBClassifier(), xgb.XGBRegressor()
        m_res.load_model(os.path.join(CARPETA_MODELOS, f"{capa}_resultado_{i}.json"))
        m_bip.load_model(os.path.join(CARPETA_MODELOS, f"{capa}_bip_{i}.json"))
        modelos.append((m_res, m_bip))
    return arch['prob'], arch['bip'], modelos

In [ ]:
OOF, MODELOS = {}, {}
for capa in VARIABLES:
    ya_existe = os.path.exists(os.path.join(CARPETA_MODELOS, f"{capa}_oof.npz"))
    if REENTRENAR or not ya_existe:
        print(f"Entrenando {capa}...")
        prob, bip, modelos = entrenar_capa(capa)
    else:
        print(f"Cargando {capa} desde disco")
        prob, bip, modelos = cargar_capa(capa)
    OOF[capa] = {'prob': prob, 'bip': bip}
    MODELOS[capa] = modelos

## Paso 5: Carreras esperadas (xRV) y escala Plus

Para cada pitcheo: $xRV = \sum_k P(k)\cdot RV_k$, con la bola en juego valuada como `(carreras esperadas del evento) − (valor del conteo)`.

Luego **invertimos el signo** (más alto = mejor para el pitcher) y estandarizamos **por temporada** contra todos los pitcheos de la liga:

$$\text{Plus} = 100 + 10 \cdot \frac{-xRV - \mu_{temporada}}{\sigma_{temporada}}$$

100 es el promedio de la LMB en esa temporada; 110 es una desviación estándar mejor. Guardamos $\mu$ y $\sigma$ para usar **la misma regla** al calificar pitcheos en otras altitudes (Paso 7).

In [ ]:
def matriz_rv(capa, bip_pred, balls, strikes, v_conteo):
    """Valor en carreras de cada una de las 5 clases, para cada pitcheo (matriz n × 5)."""
    n = len(bip_pred)
    R = np.zeros((n, len(CLASES)), dtype=np.float64)
    if capa == 'stuff':                                          # Stuff+ no conoce el conteo: promedio de liga
        for k, c in enumerate(CLASES[:-1]):
            R[:, k] = rv_clase_liga[c]
        R[:, -1] = bip_pred - v_conteo_bip_liga
    else:                                                        # Location+/Pitching+: valor según el conteo
        conteos = pd.MultiIndex.from_arrays([balls, strikes])
        for k, c in enumerate(CLASES[:-1]):
            R[:, k] = conteos.map(rv_clase_conteo.loc[c]).to_numpy(dtype=float)
        R[:, -1] = bip_pred - v_conteo
    return R


def calcular_xrv(capa, prob, bip_pred, datos=None):
    """xRV = suma de P(clase) × valor(clase). Positivo = bueno para el bateador."""
    datos = d if datos is None else datos
    R = matriz_rv(capa, bip_pred, datos['Balls'].to_numpy(), datos['Strikes'].to_numpy(), datos['v_conteo'].to_numpy())
    return (prob * R).sum(axis=1)


ESCALA = {}                                                      # μ y σ por capa y temporada
for capa in VARIABLES:
    d[f'xrv_{capa}'] = calcular_xrv(capa, OOF[capa]['prob'], OOF[capa]['bip'])
    puntaje = -d[f'xrv_{capa}']                                  # invertir: más alto = mejor para el pitcher
    ESCALA[capa] = puntaje.groupby(d['year']).agg(['mean', 'std'])
    mu = d['year'].map(ESCALA[capa]['mean'])
    sd = d['year'].map(ESCALA[capa]['std'])
    d[f'{capa}_plus'] = 100 + 10 * (puntaje - mu) / sd

# Probabilidades de Pitching+ como columnas legibles (los "targets" que lista el reto)
for k, c in enumerate(CLASES):
    d[f'p_{c}'] = OOF['pitching']['prob'][:, k]

d[['stuff_plus', 'location_plus', 'pitching_plus']].describe().round(1)

## Paso 6: Validación

### 6a. ¿Predice mejor que no saber nada?
- **Log loss** del modelo de resultado contra una **línea base**: la frecuencia de cada resultado (por conteo en Location+/Pitching+; global en Stuff+). Menor es mejor.
- **xRV contra la línea base del reto** ("xRunValue promedio por conteo"). Por cómo se construyó `rv_pitcheo`, esa línea base es ≈ 0 en todos los conteos. Reportamos RMSE y la correlación entre xRV y lo que realmente pasó.
- **Calibración por deciles:** si ordenamos los pitcheos por xRV, el valor real promedio de cada decil debe seguir la misma línea.

Todas las métricas usan predicciones **fuera de muestra por pitcher**.

In [ ]:
def linea_base_prob(por_conteo):
    """Probabilidad de cada clase sin mirar el pitcheo: frecuencia global o por conteo."""
    if not por_conteo:
        frec = d['y'].value_counts(normalize=True).sort_index().to_numpy()
        return np.tile(frec, (len(d), 1))
    frec = pd.crosstab([d['Balls'], d['Strikes']], d['y'], normalize='index')
    return frec.loc[pd.MultiIndex.from_frame(d[['Balls', 'Strikes']])].to_numpy()


real = d['rv_pitcheo'].to_numpy()
filas = []
for capa in VARIABLES:
    base = linea_base_prob(por_conteo=(capa != 'stuff'))
    xrv = d[f'xrv_{capa}'].to_numpy()
    filas.append({
        'capa': capa,
        'logloss_modelo': log_loss(d['y'], OOF[capa]['prob'], labels=range(len(CLASES))),
        'logloss_base': log_loss(d['y'], base, labels=range(len(CLASES))),
        'rmse_modelo': np.sqrt(mean_squared_error(real, xrv)),
        'rmse_base (xRV=0)': np.sqrt(np.mean(real ** 2)),
        'corr(xRV, real)': np.corrcoef(xrv, real)[0, 1],
    })
validacion = pd.DataFrame(filas).set_index('capa')
validacion['mejora_logloss_%'] = 100 * (1 - validacion['logloss_modelo'] / validacion['logloss_base'])
validacion.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, capa in zip(axes, VARIABLES):
    decil = pd.qcut(d[f'xrv_{capa}'], 10, labels=False)          # 10 grupos del mismo tamaño por xRV
    t = d.groupby(decil).agg(xrv=(f'xrv_{capa}', 'mean'), real=('rv_pitcheo', 'mean'))
    ax.plot(t['xrv'], t['real'], 'o-', label='real por decil')
    lim = [t[['xrv', 'real']].min().min(), t[['xrv', 'real']].max().max()]
    ax.plot(lim, lim, '--', color='gray', label='calibración perfecta')
    ax.set_title(f'{capa}: xRV predicho vs. real')
    ax.set_xlabel('xRV predicho (carreras/pitcheo)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('RV real promedio')
axes[0].legend()
plt.tight_layout()
plt.show()

### 6b. Estabilidad y capacidad predictiva (la prueba que usa FanGraphs)
Un buen Stuff+ debe:
1. **Ser estable:** el Stuff+ de un pitcher en juegos pares se debe parecer al de juegos impares (*split-half*).
2. **Predecir el futuro mejor que el resultado real:** el Stuff+ de 2024 debe predecir las carreras permitidas por pitcheo en 2025 **mejor que las propias carreras de 2024**. Si lo logra, está midiendo habilidad y no suerte.

Usamos pitchers con al menos 300 pitcheos en cada mitad o temporada.

In [ ]:
MIN_PITCHEOS = 300
d['rv_pitcher'] = -d['rv_pitcheo']                               # carreras evitadas (más alto = mejor)
METRICAS_PITCHER = ['stuff_plus', 'location_plus', 'pitching_plus', 'rv_pitcher']

# 1) Split-half: juegos pares vs. impares de cada pitcher-temporada
numero_juego = d.groupby(['pitcher_anon_id', 'year'])['game_anon_id'].transform(lambda s: pd.factorize(s)[0])
d['mitad'] = numero_juego % 2
mitades = d.groupby(['pitcher_anon_id', 'year', 'mitad'])[METRICAS_PITCHER].agg('mean')
n_mitad = d.groupby(['pitcher_anon_id', 'year', 'mitad']).size()
mitades = mitades[n_mitad >= MIN_PITCHEOS / 2].unstack('mitad').dropna()
estabilidad = pd.Series({m: mitades[(m, 0)].corr(mitades[(m, 1)]) for m in METRICAS_PITCHER}, name='corr_mitades')

# 2) Año t → año t+1
por_temp = d.groupby(['pitcher_anon_id', 'year'])[METRICAS_PITCHER].mean()
n_temp = d.groupby(['pitcher_anon_id', 'year']).size()
por_temp = por_temp[n_temp >= MIN_PITCHEOS].reset_index()
siguiente = por_temp.assign(year=por_temp['year'] - 1)           # el año siguiente, alineado al anterior
pares_temp = por_temp.merge(siguiente, on=['pitcher_anon_id', 'year'], suffixes=('', '_sig'))
predictivo = pd.Series({m: pares_temp[m].corr(pares_temp['rv_pitcher_sig']) for m in METRICAS_PITCHER},
                       name='corr_con_RV_año_siguiente')

print(f"Pitchers en split-half: {len(mitades)} | parejas año→año: {len(pares_temp)}")
pd.concat([estabilidad, predictivo], axis=1).round(3)

### 6c. *Season holdout* y calibración por altitud
- **Season holdout:** entrenamos con 2024–2025 y evaluamos en 2026, que el modelo nunca vio.
- **Por altitud** (sustituto del *park holdout*, porque no hay columna de estadio): el xRV promedio predicho debe coincidir con el real en cada nivel de altitud. Si el modelo estuviera mal calibrado en CDMX, aquí se vería.

In [ ]:
def season_holdout(capa, anio_prueba=2026):
    """Entrena con las otras temporadas y evalúa en anio_prueba."""
    X = d[VARIABLES[capa]]
    prueba = np.flatnonzero(d['year'].to_numpy() == anio_prueba)
    entrena = np.flatnonzero(d['year'].to_numpy() != anio_prueba)
    m_res = entrenar_modelo(X, d['y'].to_numpy(), entrena, 'resultado')
    m_bip = entrenar_modelo(X, d['rv_evento'].to_numpy(), np.intersect1d(entrena, np.flatnonzero(en_juego)), 'bip')
    prob = m_res.predict_proba(X.iloc[prueba])
    xrv = calcular_xrv(capa, prob, m_bip.predict(X.iloc[prueba]), d.iloc[prueba])
    base = linea_base_prob(capa != 'stuff')[prueba]
    return {'capa': capa, 'logloss_modelo': log_loss(d['y'].iloc[prueba], prob, labels=range(5)),
            'logloss_base': log_loss(d['y'].iloc[prueba], base, labels=range(5)),
            'corr(xRV, real)': np.corrcoef(xrv, real[prueba])[0, 1]}


archivo_sh = os.path.join(CARPETA_MODELOS, "season_holdout.csv")
if REENTRENAR or not os.path.exists(archivo_sh):
    temporada = pd.DataFrame([season_holdout(c) for c in VARIABLES]).set_index('capa')
    temporada.to_csv(archivo_sh)
else:
    temporada = pd.read_csv(archivo_sh, index_col='capa')
temporada['mejora_logloss_%'] = 100 * (1 - temporada['logloss_modelo'] / temporada['logloss_base'])
print("Season holdout (entrena 2024–2025, prueba 2026):")
display(temporada.round(4))

print("Calibración por altitud (xRV promedio predicho vs. real, carreras por 100 pitcheos):")
calib = d.groupby('altitude_category', observed=True).agg(
    pitcheos=('rv_pitcheo', 'size'), real=('rv_pitcheo', 'mean'),
    **{f'xrv_{c}': (f'xrv_{c}', 'mean') for c in VARIABLES})
display((calib.drop(columns='pitcheos') * 100).round(3).join(calib['pitcheos']))

### 6d. ¿Qué mueve el Stuff+? (valores SHAP)
Para explicarle al pitching coach qué hace bueno a un pitcheo, entrenamos un **modelo explicador**: un XGBoost que imita el xRV de Stuff+ a partir de las mismas variables. Con él calculamos los **valores SHAP**, cuánto suma o resta cada variable al xRV de cada pitcheo, usando `pred_contribs=True` de XGBoost. La importancia de cada variable es el promedio del valor absoluto de su SHAP.

In [ ]:
X_stuff = d[VARIABLES['stuff']]
muestra = d.sample(150_000, random_state=SEMILLA).index            # muestra para que sea rápido
explicador = xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, tree_method='hist',
                              max_bin=128, n_jobs=4, random_state=SEMILLA)
explicador.fit(X_stuff.loc[muestra], -d.loc[muestra, 'xrv_stuff'])   # imita el puntaje (más alto = mejor)
print(f"R² del explicador: {explicador.score(X_stuff.loc[muestra], -d.loc[muestra, 'xrv_stuff']):.3f}")

shap = explicador.get_booster().predict(xgb.DMatrix(X_stuff.loc[muestra]), pred_contribs=True)[:, :-1]
importancia = pd.Series(np.abs(shap).mean(axis=0), index=VARIABLES['stuff']).sort_values()
importancia.plot.barh(figsize=(7, 5), title='Stuff+: importancia media |SHAP|')
plt.xlabel('carreras por pitcheo')
plt.tight_layout()
plt.show()

## Paso 7: Efecto altitud: el mismo pitcheo en cada nivel

Para responder "¿qué tan bueno sería este pitcheo en CDMX?" no basta con cambiar la variable `altitud`. **La física también cambia:** en altitud el pitcheo se mueve menos. Si solo cambiáramos la etiqueta, estaríamos calificando un pitcheo imposible: el movimiento de nivel del mar en el aire de CDMX.

Por eso hacemos un **contrafactual físicamente consistente**:
1. Con la comparación **emparejada** (mismo pitcher, tipo y temporada) medimos cuánto cambian el IVB y el HB de cada tipo de pitcheo en cada nivel de altitud. Es el análisis de `IdentificadorDePitcheos.ipynb`.
2. Para calificar un pitcheo en otro nivel, **le quitamos el efecto de su altitud real y le sumamos el del nivel nuevo**, en su movimiento y en sus diferencias contra la recta (que también cambia).
3. Cambiamos `altitud` y predecimos con **el mismo modelo del bloque de ese pitcher** (sigue siendo fuera de muestra).
4. Convertimos a escala Plus con **la misma μ y σ** de la liga real, así los números son comparables entre niveles.

Solo se ajustan IVB y HB: la velocidad y el spin casi no cambian en la comparación emparejada (y sus pequeños cambios parecen más de calibración de Trackman que de física).

In [ ]:
def cambio_emparejado(col, min_n=15):
    """Cambio promedio de `col` por tipo y nivel de altitud, comparando cada pitcher-temporada-tipo
    contra sí mismo sin altitud. Promedio ponderado por tamaño de muestra."""
    llave = ['pitcher_anon_id', 'year', 'tipo']
    g = d.groupby(llave + ['altitud'], observed=True)[col].agg(['mean', 'size']).reset_index()
    base = g[(g['altitud'] == 0) & (g['size'] >= min_n)]
    salida = {}
    for nivel in (1, 2):
        alt = g[(g['altitud'] == nivel) & (g['size'] >= min_n)]
        p = alt.merge(base, on=llave, suffixes=('_alt', '_base'))
        p['w'] = 1 / (1 / p['size_alt'] + 1 / p['size_base'])
        p['delta'] = p['mean_alt'] - p['mean_base']
        salida[nivel] = p.groupby('tipo', observed=True).apply(
            lambda x: np.average(x['delta'], weights=x['w']), include_groups=False)
    tabla = pd.DataFrame(salida).reindex(TIPOS_MODELO)
    tabla.insert(0, 0, 0.0)                                      # nivel 0 = referencia, cambio cero
    return tabla


DELTA = {'InducedVertBreak': cambio_emparejado('InducedVertBreak'), 'hb': cambio_emparejado('hb')}
print("Cambio de IVB (pulgadas) por tipo y nivel de altitud:")
display(DELTA['InducedVertBreak'].round(2))
print("Cambio de HB hacia el brazo (pulgadas):")
display(DELTA['hb'].round(2))

In [ ]:
def mover_a_altitud(datos, nivel_nuevo, mover_fisica=True, mover_ambiente=True):
    """Copia de las variables con el pitcheo 'trasladado' a otro nivel de altitud.
    mover_fisica: ajusta el movimiento (IVB, HB y sus diferencias contra la recta).
    mover_ambiente: cambia la variable `altitud` (bateo, vuelo de la bola, todo lo que no es movimiento)."""
    x = datos.copy()
    if not mover_fisica:
        x['altitud'] = nivel_nuevo if mover_ambiente else datos['altitud']
        return x
    tipo = datos['tipo'].astype(str)
    fb = datos['fb_tipo'].fillna('Four-Seam').astype(str)        # si no hay recta primaria, se asume Four-Seam
    nivel_real = datos['altitud']
    for col, diff_col in [('InducedVertBreak', 'ivb_diff_vs_fb'), ('hb', 'hb_diff_vs_fb')]:
        tabla = DELTA[col].stack()                               # índice (tipo, nivel) → cambio
        buscar = lambda t, n: pd.MultiIndex.from_arrays([t, n]).map(tabla).to_numpy(dtype=float)
        ajuste_pitcheo = buscar(tipo, np.full(len(x), nivel_nuevo)) - buscar(tipo, nivel_real)
        ajuste_recta = buscar(fb, np.full(len(x), nivel_nuevo)) - buscar(fb, nivel_real)
        x[col] = datos[col] + ajuste_pitcheo                     # el pitcheo cambia su movimiento
        x[diff_col] = datos[diff_col] + ajuste_pitcheo - ajuste_recta   # y su diferencia contra la recta
    x['altitud'] = nivel_nuevo if mover_ambiente else datos['altitud']
    return x


def calificar_en_altitud(capa, nivel, **opciones):
    """Plus de cada pitcheo si se hubiera lanzado en `nivel`, con el modelo de su propio bloque."""
    datos = mover_a_altitud(d, nivel, **opciones)
    X = datos[VARIABLES[capa]]
    prob = np.zeros((len(d), len(CLASES)))
    bip = np.zeros(len(d))
    for (m_res, m_bip), (_, prueba) in zip(MODELOS[capa], bloques):
        prob[prueba] = m_res.predict_proba(X.iloc[prueba])
        bip[prueba] = m_bip.predict(X.iloc[prueba])
    puntaje = -calcular_xrv(capa, prob, bip)
    return 100 + 10 * (puntaje - d['year'].map(ESCALA[capa]['mean'])) / d['year'].map(ESCALA[capa]['std'])


NIVELES = {0: 'nivel_mar', 1: 'media', 2: 'cdmx'}
for capa in ['stuff', 'pitching']:
    for nivel, nombre in NIVELES.items():
        d[f'{capa}_plus_{nombre}'] = calificar_en_altitud(capa, nivel)
        print(f"{capa} calificado en {nombre}")

# Descomposición: el pitcheo con el movimiento de CDMX pero el ambiente de nivel del mar.
# Primero lo llevamos todo a nivel del mar y desde ahí movemos solo la física a CDMX.
nivel_mar = mover_a_altitud(d, 0)
solo_fisica = mover_a_altitud(nivel_mar.assign(altitud=0), 2, mover_ambiente=False)
X = solo_fisica[VARIABLES['stuff']]
prob, bip = np.zeros((len(d), len(CLASES))), np.zeros(len(d))
for (m_res, m_bip), (_, prueba) in zip(MODELOS['stuff'], bloques):
    prob[prueba] = m_res.predict_proba(X.iloc[prueba])
    bip[prueba] = m_bip.predict(X.iloc[prueba])
d['stuff_plus_cdmx_solo_fisica'] = 100 + 10 * (-calcular_xrv('stuff', prob, bip) - d['year'].map(ESCALA['stuff']['mean'])) \
                                   / d['year'].map(ESCALA['stuff']['std'])
print("stuff calificado con física de CDMX y ambiente de nivel del mar")

### 7a. Efecto altitud por tipo de pitcheo
Cada pitcheo de la base se califica en los tres niveles. La diferencia **CDMX − nivel del mar** es cuánto Stuff+ pierde (o gana) ese tipo de pitcheo **solo por el aire**, con el mismo pitcher y la misma mecánica.

Y lo **descomponemos** en dos partes:
- **Δ física:** el pitcheo con el movimiento de CDMX, pero en ambiente de nivel del mar. Es lo que se pierde porque el pitcheo se mueve menos.
- **Δ ambiente:** lo que queda. Es el efecto de la altitud sobre el contacto (la bola vuela más y se batea más duro) con el pitcheo ya ajustado.

In [ ]:
efecto_tipo = d.groupby('tipo', observed=True).agg(
    pitcheos=('stuff_plus', 'size'),
    stuff_nivel_mar=('stuff_plus_nivel_mar', 'mean'),
    stuff_media=('stuff_plus_media', 'mean'),
    stuff_cdmx=('stuff_plus_cdmx', 'mean'),
    pitching_nivel_mar=('pitching_plus_nivel_mar', 'mean'),
    pitching_cdmx=('pitching_plus_cdmx', 'mean'),
)
efecto_tipo['Δ stuff (cdmx − mar)'] = efecto_tipo['stuff_cdmx'] - efecto_tipo['stuff_nivel_mar']
solo_fis = d.groupby('tipo', observed=True)['stuff_plus_cdmx_solo_fisica'].mean()
efecto_tipo['  Δ física'] = solo_fis - efecto_tipo['stuff_nivel_mar']
efecto_tipo['  Δ ambiente'] = efecto_tipo['stuff_cdmx'] - solo_fis
efecto_tipo['Δ pitching (cdmx − mar)'] = efecto_tipo['pitching_cdmx'] - efecto_tipo['pitching_nivel_mar']
efecto_tipo.sort_values('Δ stuff (cdmx − mar)').round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
orden = efecto_tipo.sort_values('Δ stuff (cdmx − mar)').index
x = np.arange(len(orden))
for i, (nombre, color) in enumerate([('nivel_mar', 'tab:blue'), ('media', 'tab:orange'), ('cdmx', 'tab:red')]):
    ax.bar(x + (i - 1) * 0.27, efecto_tipo.loc[orden, f'stuff_{nombre}'] - 100, width=0.27,
           bottom=100, color=color, label=nombre)
ax.axhline(100, color='gray', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(orden)
ax.set_ylabel('Stuff+ promedio')
ax.set_title('Mismo pitcheo calificado en cada nivel de altitud')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 7b. Comprobación con datos observados
El contrafactual es un modelo. Lo contrastamos con lo que **realmente** pasó: el Stuff+ del mismo pitcher-temporada-tipo en juegos de altitud extrema vs. sin altitud (mínimo 15 pitcheos por lado). Si los dos métodos coinciden en dirección y orden, el efecto es robusto.

In [ ]:
llave = ['pitcher_anon_id', 'year', 'tipo']
g = d.groupby(llave + ['altitud'], observed=True)['stuff_plus'].agg(['mean', 'size']).reset_index()
emp = g[(g['altitud'] == 2) & (g['size'] >= 15)].merge(g[(g['altitud'] == 0) & (g['size'] >= 15)],
                                                      on=llave, suffixes=('_cdmx', '_mar'))
emp['w'] = 1 / (1 / emp['size_cdmx'] + 1 / emp['size_mar'])
observado = emp.groupby('tipo', observed=True).apply(
    lambda x: pd.Series({'pares': len(x),
                         'Δ observado': np.average(x['mean_cdmx'] - x['mean_mar'], weights=x['w'])}),
    include_groups=False)
observado['Δ contrafactual'] = efecto_tipo['Δ stuff (cdmx − mar)']
observado.round(1)

## Paso 8: Agregación y respuestas del entregable

### 8a. Por pitcher-temporada y tipo de pitcheo

**¿Qué columna usar?**
| Columna | Qué significa | Para qué sirve |
|---|---|---|
| `stuff_plus` | calificado donde realmente lanzó | describir lo que pasó |
| `stuff_plus_nivel_mar` | todos sus pitcheos llevados a nivel del mar | **comparar pitchers de forma justa** (neutral al estadio) |
| `stuff_plus_cdmx` | todos sus pitcheos llevados a CDMX | **decidir para Diablos**: cómo rendiría en el Harp Helú |

El 100 es el promedio de todos los pitcheos reales de la liga (mezcla de altitudes). Por eso a nivel del mar el promedio queda arriba de 100 y en CDMX abajo: es el efecto altitud.

Mínimo 50 pitcheos del tipo.

In [ ]:
COLS_PLUS = ['stuff_plus', 'location_plus', 'pitching_plus',
             'stuff_plus_nivel_mar', 'stuff_plus_media', 'stuff_plus_cdmx', 'stuff_plus_cdmx_solo_fisica',
             'pitching_plus_nivel_mar', 'pitching_plus_cdmx']
COLS_PROB = [f'p_{c}' for c in CLASES]

por_pitcher_tipo = (d.groupby(['pitcher_anon_id', 'year', 'tipo'], observed=True)
                     .agg(pitcheos=('stuff_plus', 'size'), uso_pct=('uso_pct', 'first'),
                          velo=('RelSpeed', 'mean'), ivb=('InducedVertBreak', 'mean'), hb=('hb', 'mean'),
                          **{c: (c, 'mean') for c in COLS_PLUS + COLS_PROB})
                     .query('pitcheos >= 50'))
por_pitcher_tipo['Δ stuff altitud'] = por_pitcher_tipo['stuff_plus_cdmx'] - por_pitcher_tipo['stuff_plus_nivel_mar']
top = por_pitcher_tipo.sort_values('stuff_plus_cdmx', ascending=False).head(15)
top.round(1).assign(**{c: top[c].round(3) for c in COLS_PROB + ['uso_pct']})   # probabilidades con 3 decimales

### 8b. Por pitcher-temporada
Promedio de todos sus pitcheos (ponderado por uso, porque cada pitcheo cuenta una vez). Mínimo 300 pitcheos. **`Δ stuff altitud`** indica qué tanto le afecta lanzar en CDMX: los más negativos son los más afectados y los cercanos a 0 los más consistentes.

In [ ]:
por_pitcher = (d.groupby(['pitcher_anon_id', 'year'])
                .agg(pitcheos=('stuff_plus', 'size'), mano=('PitcherThrows', 'first'),
                     **{c: (c, 'mean') for c in COLS_PLUS})
                .query('pitcheos >= 300'))
por_pitcher['Δ stuff altitud'] = por_pitcher['stuff_plus_cdmx'] - por_pitcher['stuff_plus_nivel_mar']

print("Mejor Stuff+ calificado en CDMX:")
display(por_pitcher.sort_values('stuff_plus_cdmx', ascending=False).head(10).round(1))
print("Más afectados por la altitud:")
display(por_pitcher.sort_values('Δ stuff altitud').head(10).round(1))
print("Más consistentes (menor pérdida):")
display(por_pitcher.sort_values('Δ stuff altitud', ascending=False).head(10).round(1))

### 8c. ¿Qué arsenal produce el Stuff+ más alto en CDMX?
**Arsenal** = combinación de tipos que el pitcher usa al menos 10% del tiempo (por ejemplo "Four-Seam + Slider + Changeup"). Para cada arsenal promediamos el Stuff+ **calificado en CDMX** de los pitchers que lo usan. Solo arsenales con al menos 8 pitcher-temporadas.

In [ ]:
USO_MINIMO = 0.10
uso = por_pitcher_tipo.reset_index()
uso = uso[uso['uso_pct'] >= USO_MINIMO]
arsenal = (uso.sort_values('tipo')
              .groupby(['pitcher_anon_id', 'year'])['tipo']
              .agg(lambda s: ' + '.join(s.astype(str)))
              .rename('arsenal'))
por_pitcher = por_pitcher.join(arsenal)

por_arsenal = (por_pitcher.dropna(subset=['arsenal'])
               .groupby('arsenal')
               .agg(pitcher_temporadas=('stuff_plus', 'size'),
                    stuff_cdmx=('stuff_plus_cdmx', 'mean'), stuff_nivel_mar=('stuff_plus_nivel_mar', 'mean'),
                    pitching_cdmx=('pitching_plus_cdmx', 'mean'))
               .query('pitcher_temporadas >= 8'))
por_arsenal['Δ stuff altitud'] = por_arsenal['stuff_cdmx'] - por_arsenal['stuff_nivel_mar']
por_arsenal.sort_values('stuff_cdmx', ascending=False).round(1)

### 8d. Exportar resultados
- `resultados/stuff_plus_por_pitcheo.pkl`: cada pitcheo con sus tres Plus, las probabilidades y los contrafactuales de altitud (pickle, porque son 600 mil renglones).
- `resultados/*.csv`: tablas por pitcher-tipo, por pitcher, por arsenal y efecto por tipo, listas para Excel o para el dashboard.

In [ ]:
cols_pitcheo = ['PitchUID', 'year', 'game_anon_id', 'pitcher_anon_id', 'tipo', 'altitude_category',
                'Balls', 'Strikes', 'RelSpeed', 'InducedVertBreak', 'hb', 'loc_x', 'PlateLocHeight',
                'xrv_stuff', 'xrv_location', 'xrv_pitching', 'rv_pitcheo'] + COLS_PLUS + COLS_PROB
d[cols_pitcheo].to_pickle(os.path.join(CARPETA_RESULTADOS, "stuff_plus_por_pitcheo.pkl"))
por_pitcher_tipo.round(3).to_csv(os.path.join(CARPETA_RESULTADOS, "stuff_plus_por_pitcher_tipo.csv"))
por_pitcher.round(3).to_csv(os.path.join(CARPETA_RESULTADOS, "stuff_plus_por_pitcher.csv"))
por_arsenal.round(3).to_csv(os.path.join(CARPETA_RESULTADOS, "stuff_plus_por_arsenal.csv"))
efecto_tipo.round(3).to_csv(os.path.join(CARPETA_RESULTADOS, "efecto_altitud_por_tipo.csv"))
print("Archivos en resultados/:", sorted(os.listdir(CARPETA_RESULTADOS)))

## Paso 9: Subir los resultados al API 2

La web app no lee archivos del notebook: lee el **API 2**. Esta celda sube:
- `stuff_plus_por_pitcher_tipo`: el Stuff+ de cada pitcher × temporada × tipo en los 3 niveles de altitud. Cada parque de la app toma el de su nivel.
- `validacion`: las métricas de los pasos 6a, 6b y 6c, para la página de Metodología.

Luego le avisa a la web app que recargue. **Antes:** prender todo con `bash apis/iniciar_todo.sh` desde la raíz del repo. Cada vez que cambies el modelo, sube `VERSION_MODELO` (v2, v3...): la versión anterior queda guardada en el historial del API 2.

Necesita `polars`, `pyarrow` y `requests` en el Python de este notebook. Si falta alguno: `%pip install polars pyarrow requests`.

In [ ]:
# --- Paso 9: subir los resultados al API 2 (de ahí los toma la web app) ---
# Antes, en la Terminal, desde la raíz del repo:  bash apis/iniciar_todo.sh
import sys, requests


def buscar_repo():
    """La carpeta del repo: subiendo desde donde corre el notebook hasta encontrar apis/api/cliente.py."""
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "apis", "api", "cliente.py")):
            return p
        p = os.path.dirname(p)
    return os.path.expanduser("~/Desktop/warroom")


REPO = buscar_repo()
sys.path.insert(0, os.path.join(REPO, "apis", "api"))           # cliente.py vive en apis/api/
from cliente import ClienteAPI2

VERSION_MODELO = "v1"      # cámbiala cada vez que subas una versión nueva (v2, v3... o la fecha)


def leer_env(ruta):
    """Lee un .env (CLAVE=valor) como diccionario."""
    with open(ruta) as f:
        return dict(l.strip().split("=", 1) for l in f if "=" in l and not l.lstrip().startswith("#"))


api2 = ClienteAPI2("http://localhost:8001", leer_env(os.path.join(REPO, "apis", "api2", ".env"))["API_KEY_ESCRITURA"])

# 1) Stuff+ por pitcher × temporada × tipo, calificado en los 3 niveles de altitud (CONTRATO_API2.md)
tabla_stuff = por_pitcher_tipo.reset_index()[['pitcher_anon_id', 'year', 'tipo', 'pitcheos',
                                              'stuff_plus_nivel_mar', 'stuff_plus_media', 'stuff_plus_cdmx']]
tabla_stuff['tipo'] = tabla_stuff['tipo'].astype(str)
print(api2.subir("stuff_plus_por_pitcher_tipo", tabla_stuff, modelo="stuff_plus", version=VERSION_MODELO))

# 2) Métricas de validación de los pasos 6a, 6b y 6c (página Metodología de la web app)
GKF = 'GroupKFold por pitcher (5 bloques)'
filas = []
for capa, nombre in {'stuff': 'Stuff+', 'location': 'Location+', 'pitching': 'Pitching+'}.items():
    filas += [
        {'nombre': f'{nombre} · mejora de log loss vs. línea base (%)', 'valor': validacion.loc[capa, 'mejora_logloss_%'], 'split': GKF},
        {'nombre': f'{nombre} · corr(xRV, RV real)', 'valor': validacion.loc[capa, 'corr(xRV, real)'], 'split': GKF},
        {'nombre': f'{nombre} · split-half r', 'valor': estabilidad[f'{capa}_plus'], 'split': 'juegos pares vs. impares'},
        {'nombre': f'{nombre} · corr con RV del año siguiente', 'valor': predictivo[f'{capa}_plus'], 'split': 'año t → año t+1'},
        {'nombre': f'{nombre} · mejora de log loss en 2026 (%)', 'valor': temporada.loc[capa, 'mejora_logloss_%'],
         'split': 'season holdout: entrena 2024–2025, prueba 2026'},
        {'nombre': f'{nombre}: resultado del pitcheo', 'tipo': 'submodelo', 'metrica': 'log loss',
         'objetivo': 'bola / strike cantado / whiff / foul / en juego', 'valor': validacion.loc[capa, 'logloss_modelo']},
    ]
filas.append({'nombre': 'Referencia: RV real · corr con RV del año siguiente', 'valor': predictivo['rv_pitcher'],
              'split': 'año t → año t+1'})
tabla_val = pd.DataFrame(filas).fillna({'tipo': 'metrica', 'split': '', 'metrica': '', 'objetivo': ''})
tabla_val['valor'] = tabla_val['valor'].astype(float).round(4)
print(api2.subir("validacion", tabla_val, modelo="stuff_plus", version=VERSION_MODELO))

# 3) Avisarle a la web app que hay resultados nuevos
try:
    r = requests.post("http://localhost:8002/api/admin/reload", timeout=600)
    print("✓ Web app usando:", r.json()["model"]["name"])
except requests.RequestException:
    print("La web app no está prendida; cuando la prendas ya usará este modelo.")
